# 00 — Base Model Evaluation (BEFORE Fine-tuning)

**Purpose:** Record baseline BLEU / ROUGE / F1 for both models BEFORE any fine-tuning.
This is your 'before' column in the comparison table. Run this FIRST before any training.

**Models evaluated:**
- `unsloth/Qwen3-8B-unsloth-bnb-4bit`
- `unsloth/Llama-3.2-3B-Instruct-bnb-4bit`

In [ ]:
!pip install unsloth transformers datasets sacrebleu rouge-score -q

In [ ]:
import sys
sys.path.append('/kaggle/working/final_year_proj_finetuning')

from src.data_utils import load_jsonl
from src.evaluation import evaluate_translation, evaluate_summarization, evaluate_qa, build_results_table
from src.training_utils import load_model
from src.utils import set_seed, print_gpu_memory
import pandas as pd

set_seed(42)
print_gpu_memory()

all_results = {}

In [ ]:
# Load val sets
val_tr  = load_jsonl('data/processed/translation_val.jsonl')
val_sum = load_jsonl('data/processed/summarization_val.jsonl')
val_qa  = load_jsonl('data/processed/qa_val.jsonl')

In [ ]:
# ── EVALUATE QWEN3-8B BASE ───────────────────
print('\n' + '='*60)
print('QWEN3-8B BASE MODEL')
print('='*60)

model, tokenizer = load_model('qwen3-8b')

tr_res  = evaluate_translation(model, tokenizer, val_tr, n_samples=100)
sum_res = evaluate_summarization(model, tokenizer, val_sum, n_samples=100)
qa_res  = evaluate_qa(model, tokenizer, val_qa, n_samples=100)

all_results['Qwen3-8B Base'] = {
    'BLEU': tr_res['bleu'],
    'ROUGE-1': sum_res['rouge1'],
    'ROUGE-L': sum_res['rougeL'],
    'QA F1': qa_res['f1'],
    'QA EM': qa_res['exact_match'],
}

print('\nQwen3-8B Base Results:', all_results['Qwen3-8B Base'])

# Free memory before next model
del model
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# ── EVALUATE LLAMA-3.2-3B BASE ───────────────
print('\n' + '='*60)
print('LLAMA-3.2-3B BASE MODEL')
print('='*60)

model, tokenizer = load_model('llama32-3b')

tr_res  = evaluate_translation(model, tokenizer, val_tr, n_samples=100)
sum_res = evaluate_summarization(model, tokenizer, val_sum, n_samples=100)
qa_res  = evaluate_qa(model, tokenizer, val_qa, n_samples=100)

all_results['Llama-3.2-3B Base'] = {
    'BLEU': tr_res['bleu'],
    'ROUGE-1': sum_res['rouge1'],
    'ROUGE-L': sum_res['rougeL'],
    'QA F1': qa_res['f1'],
    'QA EM': qa_res['exact_match'],
}

print('\nLlama-3.2-3B Base Results:', all_results['Llama-3.2-3B Base'])

del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# ── SAVE BASELINE RESULTS ────────────────────
df = build_results_table(all_results)
print('\nBASELINE RESULTS TABLE')
print(df.to_string(index=False))

df.to_csv('results/base_vs_finetuned/baseline_results.csv', index=False)
print('\nSaved to results/base_vs_finetuned/baseline_results.csv')